# Getting started — Financial Sentiment

Classifies financial news and tweets as negative, neutral, or positive.

**Labels:** `negative, neutral, positive`

This notebook deploys the Sigmodata SageMaker model package from AWS Marketplace, sends example texts, prints predictions, then deletes the endpoint.

**Before you run this:**
1. Subscribe on AWS Marketplace (**Continue to Subscribe** → **Continue to Configuration**).
2. Copy the **Model Package ARN** from that page into the next cell. That ARN is AWS's copy of the package (unique per region), not the seller ARN.
3. Run from a SageMaker Notebook / Studio session (execution role auto-detected), or set `role` manually.

Product page: https://www.sigmodata.com/products/financial-sentiment  
Download this notebook: https://www.sigmodata.com/products/financial-sentiment/getting-started.ipynb

In [ ]:
# Product ARN from AWS Marketplace → Continue to Configuration.
# That is AWS's copy of the model package (unique per region). Do not use a
# seller-account package ARN.
MODEL_PACKAGE_ARN = ""
assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), (
    "Paste the Model Package ARN from Marketplace Continue to Configuration"
)

## 1. Deploy a real-time endpoint

Persistent endpoints bill per hour while running. For bulk files, skip to **Batch transform** — you only pay for job runtime.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import ModelPackage

session = sagemaker.Session()
role = get_execution_role()

model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large")

## 2. Classify example texts

**Request:** `{"texts": ["...", "..."]}`  
**Response:** `{"predictions": [{"label": "...", "score": 0.94}, ...]}` — one prediction per input, in order.

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

texts = [
    "The company reported strong revenue growth exceeding analyst expectations",
    "Shares fell sharply after the profit warning issued this morning",
    "The board approved a modest dividend increase for shareholders",
]

result = predictor.predict({"texts": texts})
for text, pred in zip(texts, result["predictions"]):
    print(f"{text!r:70s} -> {pred['label']:30s} ({pred['score']:.4f})")

## 3. Clean up the endpoint

Real-time endpoints keep billing until deleted. Always run this when finished.

In [ ]:
predictor.delete_endpoint()

## Batch transform (optional)

Cheaper for bulk jobs: CSV in S3 (one text per line, no header) → `label,score` CSV out.

In [ ]:
import boto3

bucket = session.default_bucket()
input_key = "sigmodata-example/financial-sentiment/texts.csv"

sample_texts = "\n".join(texts) + "\n"
boto3.client("s3").put_object(Bucket=bucket, Key=input_key, Body=sample_texts.encode())

input_s3_uri = f"s3://{bucket}/{input_key}"
output_s3_uri = f"s3://{bucket}/sigmodata-example/financial-sentiment/output/"
print("Input:", input_s3_uri)
print("Output:", output_s3_uri)

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_s3_uri,
    strategy="MultiRecord",
    max_payload=6,
)

transformer.transform(
    data=input_s3_uri,
    content_type="text/csv",
    split_type="Line",
    wait=True,
)

In [ ]:
output_key = f"{input_key.rsplit('/', 1)[-1]}.out"
obj = boto3.client("s3").get_object(
    Bucket=bucket, Key=f"sigmodata-example/financial-sentiment/output/{output_key}"
)
print(obj["Body"].read().decode())

## Next steps

- Standalone scripts: `deploy_and_predict.py` and `batch_transform.py` in this folder.
- Product page with live snippets: https://www.sigmodata.com/products/financial-sentiment
- Support: support@sigmodata.com